# Prolog in SCRIP: the core rungs

### asserta_assertz_directive_1

In [1]:
%%prolog
:- initialization(main).
:- assertz(item(b)).
:- assertz(item(c)).
:- asserta(item(a)).

main :-
    item(X),
    write(X), nl,
    fail.
main.

a
b
c


### clause_source_record__csr_listing_names

In [2]:
%%prolog
:- use_module(library(lists)).
:- initialization(main).
:- op(700, xfx, rplus).
:- dynamic(dummy/4).

goal_expansion(L rplus R, R + L).

dummy(A,B,C,D) :- B rplus A, D rplus C.

dummy(A,B) --> { B rplus A }.

dummy((Head --> Goals), Clause, _, _) :-
        list_goal(Goals, Body),
        expand_term((Head --> Body), Clause).

dummy(g(Goal), []) --> [{Goal}, !].

dummy(Foo, _Bar, _, Baz) :-
        Baz = [Foo|_].

main :- listing(dummy/4).

:- dynamic dummy/4.

dummy(A, B, C, D) :-
    A+B,
    C+D.
dummy(A, B, C, D) :-
    A+B,
    D=C.
dummy((Head-->Goals), Clause, _, _) :-
    list_goal(Goals, Body),
    expand_term((Head-->Body), Clause).
dummy(g(Goal), [], A, B) :-
    A=[{Goal}, !|B].
dummy(Foo, _Bar, _, Baz) :-
    Baz=[Foo|_].



### writeq_format_directive_3

In [3]:
%%prolog
:- initialization(main).
main :- write('hello world'), nl, writeq('hello world'), nl, print(foo), nl, write_canonical([a,b]), nl, writeln(done), tab(3), write(x), nl, format('~w ~a ~d~n', [alpha, beta, 42]).

hello world
'hello world'
foo
'.'(a,'.'(b,[]))
done
   x
alpha beta 42


### asserta_order_1

In [4]:
%%prolog
:- dynamic(g/1).
:- initialization(main).
main :- assertz(g(1)), assertz(g(2)), asserta(g(0)), findall(X, g(X), L), write(L), nl.

[0,1,2]


### op_ite_directive_replace_2

In [5]:
%%prolog
:- op(200, fy, ~).
:- initialization(main).
main :-
    ( ~ ~ a == ~(~(a)) -> write(yes) ; write(no) ), nl,
    ( X = ~ foo, X == ~(foo) -> write(yes) ; write(no) ), nl,
    ( ~ (b + c) == ~(b + c) -> write(yes) ; write(no) ), nl.

yes
yes
yes


### call_directive_replace_3

In [6]:
%%prolog
% rung33_bridge_callN/05_call2_compound_g — call/2 where G is already a compound.
% call(G, ExtraArg) where G=succ(3) reconstructs succ(3, R) — G's args
% prepended, extra args appended. Tests the compound-G case of call/N.
:- initialization(main).
main :-
    G = succ(3),
    call(G, R),
    write(R), nl.

4


### op_ite_format_1

In [7]:
%%prolog
:- initialization(main).
main :-
    op(600, xfx, rt_infix),
    op(300, fy, rt_pre),
    ( current_op(P1, T1, rt_infix) -> format("~w ~w~n", [P1, T1]) ; write(none), nl ),
    ( current_op(P2, T2, rt_pre) -> format("~w ~w~n", [P2, T2]) ; write(none), nl ),
    op(400, yfx, rt_multi),
    ( current_op(400, yfx, rt_multi) -> write(yes) ; write(no) ), nl.

600 xfx
300 fy
yes


### undo__u01_undo_or_branch

In [8]:
%%prolog
:- initialization(main).
:- dynamic a/1.
main :-
    (   assert(a(1), Ref), undo(erase(Ref)), fail
    ;   ( a(1) -> write(still_there) ; write(erased) ), nl
    ),
    (   X = 1, undo(write(kept(X))), fail
    ;   write(after), nl
    ),
    halt.

erased
kept(1)after


### dcg_ite_list_replace_1

In [9]:
%%prolog
:- initialization(main).

digits([D|Ds]) --> digit(D), digits(Ds).
digits([]) --> [].
digit(D) --> [D], { D >= 0'0, D =< 0'9 }.

main :-
    atom_codes('123', Codes),
    ( phrase(digits(Ds), Codes) ->
        atom_codes(A, Ds), write(A)
    ; write(fail)
    ), nl.

123


### streams__st04_socket_handle_shape_and_errors

In [10]:
%%prolog
:- use_module(library(socket)).
:- initialization(main, main).
main :-
    tcp_socket(S), ( S = '$socket'(_) -> writeln(handle_is_dollar_socket) ; writeln(handle_is_a_blob) ),
    tcp_setopt(S, reuseaddr), tcp_setopt(S, nodelay),
    catch(tcp_setopt(S, bogus), error(F1, _), true), print(F1), nl,
    tcp_close_socket(S),
    catch(tcp_open_socket(not_a_socket, _), error(F2, _), true), print(F2), nl,
    tcp_socket(R), catch(tcp_connect(R, '127.0.0.1':1), error(F3, C3), true),
    ( C3 = tcp_connect/2 -> writeln(context_unifies) ; writeln(context_differs) ), F3 = socket_error(Code3, _), print(Code3), nl, tcp_close_socket(R),
    tcp_socket(B1), tcp_bind(B1, '127.0.0.1':P), tcp_listen(B1, 5), tcp_socket(B2),
    catch(tcp_bind(B2, '127.0.0.1':P), error(F4, _), true), F4 = socket_error(Code4, _), print(Code4), nl, tcp_close_socket(B2), tcp_close_socket(B1),
    gethostname(H), ( atom(H) -> writeln(hostname_atom) ; writeln(hostname_bad) ),
    tcp_host_to_address(localhost, A), print(A), nl,
    tcp_socket(F), tcp_open_socket(F, Pair), ( is_stream(Pair) -> writeln(fresh_open_ok) ; writeln(fresh_open_bad) ), close(Pair),
    halt.

handle_is_a_blob
domain_error(socket_option,bogus)
type_error(socket,not_a_socket)
context_unifies
econnrefused
eaddrinuse
hostname_atom
ip(127,0,0,1)
fresh_open_ok


### ladder__rung10_monotonic_tabling_propagates_an_assert_into_its_dependents

In [11]:
%%prolog
:- dynamic da/1 as monotonic.
:- table pa/1 as monotonic.
pa(X) :- da(Y), X is Y+1.
:- dynamic dx/1 as (incremental,monotonic).
:- table px/1 as monotonic.
:- table qx/1 as incremental.
px(X) :- dx(Y), X is Y+1.
qx(X) :- px(Y), X is Y*2.
:- table cm/2 as monotonic.
:- dynamic lm/2 as monotonic.
cm(X, Y) :- cm(Y, X).
cm(X, Z) :- cm(X, Y), cm(Y, Z).
cm(X, Y) :- lm(X, Y).
main :-
    ( pa(_) -> writeln(pa_nonempty) ; writeln(pa_empty) ),
    assert(da(0)),
    ( get_returns_for_call(pa(_), pa(1)) -> writeln(ret_pa1) ; writeln(no_ret_pa1) ),
    findall(X, pa(X), L1), writeq(L1), nl,
    ( incr_is_invalid(pa(_)) -> writeln(pa_invalid) ; writeln(pa_valid) ),
    assert(dx(1)), qx(Q1), writeq(q(Q1)), nl,
    assert(dx(2)),
    ( incr_is_invalid(qx(_)) -> writeln(qx_invalid) ; writeln(qx_valid) ),
    findall(Q, qx(Q), QL), sort(QL, QS), writeq(QS), nl,
    ignore(cm(_,_)),
    assert(lm(a, b)), assert(lm(a, c)),
    findall(X-Y, cm(X,Y), Pairs), length(Pairs, Len), writeq(len(Len)), nl,
    sort(Pairs, SP), writeq(SP), nl.
:- initialization(main).

pa_empty
ret_pa1
[1]
pa_valid
q(4)
qx_invalid
[4,6]
len(9)
[a-a,a-b,a-c,b-a,b-b,b-c,c-a,c-b,c-c]


### typetest_directive_1

In [12]:
%%prolog
:- initialization(main).
main :- a == a, a \== b, a @< b, b @> a, a @=< a, b @>= a, compare(O, 1, 2), atom(foo), number(3), integer(3), float(1.5), atomic(foo), var(_), nonvar(foo), compound(f(x)), callable(foo), write(O), nl, write(ok), nl.

<
ok
